In the previous notebook 1models_statenisland.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a better optimal model.

Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. 

# Import Packages

In [9]:
!pip install optuna

!pip install plotly ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


# Prophet

## Import the data

In [10]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=4)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date'])

# Start by cutting off data before 2020-01-06 and after 2025-03-02.
rs = rs[rs['created_date'] < '2025-03-02']
rs = rs[rs['created_date'] >= '2020-01-06']

## Restrict to STATEN ISLAND
rs = rs[rs['borough'] == 'STATEN ISLAND']

## Drop the column with borough
rs = rs.drop(columns=['borough'])

## Resample to weekly
rs_weekly = (
    rs.set_index('created_date')
      .resample('W')['count']
      .mean()
      .fillna(0)
      .reset_index()
      .rename(columns={'count': 'weekly_avg'})
)

## Rename columns for Prophet
rs_weekly.rename(columns={'created_date': 'ds', 'weekly_avg': 'y'}, inplace=True)


## Prepare Prophet

In [11]:
date_range = pd.date_range(start="2020-01-06", end="2026-03-02")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Snap each holiday to its week-ending Sunday to match rs_weekly
holidays_weekly = pd.to_datetime(holidays).to_series().apply(
    lambda d: d + pd.offsets.Week(weekday=6) if d.weekday() != 6 else d
).reset_index(drop=True)

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': holidays_weekly,
    'lower_window': 0,
    'upper_window': 1
})

# Drop duplicate weeks (two holidays can fall in the same week)
federal_holidays = federal_holidays.drop_duplicates(subset='ds')

holidays = federal_holidays

In [12]:
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd_daily = nd
else:
    wd_daily = pd.DataFrame(data["daily"])
    wd_daily["date"] = pd.to_datetime(wd_daily["time"])
    wd_daily = wd_daily.set_index("date")

# Resample to weekly (ending Sunday, to match rs_weekly)
wd = wd_daily.resample('W').agg({
    'temperature_2m_max':        'max',
    'temperature_2m_min':        'min',
    'temperature_2m_mean':       'mean',
    'apparent_temperature_max':  'max',
    'apparent_temperature_min':  'min',
    'apparent_temperature_mean': 'mean',
    'precipitation_sum':         'sum',
    'snowfall_sum':              'sum',
})

In [13]:
rs_weekly_saved = rs_weekly.copy()
df = rs_weekly.copy()

## Optuna for Hyperparameter Tuning for Prophet

In [14]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [15]:
init_weeks = '1456 days'   # 208 weeks in days
cv_period = '14 days'      # 2 weeks in days
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1],
    'seasonality_prior_scale': [5],
}

all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []
performance = []

for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)
    df_cv = cross_validation(m, initial=init_weeks, period=cv_period, horizon=forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=1)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

  0%|          | 0/30 [00:00<?, ?it/s]

In [16]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,14 days,0.3545,0.5954,0.4645,0.2618,0.1664,0.2218,0.8667


## Train the Model

In [17]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=4, freq='W')
forecast = m.predict(future)

## Plots and Evaluation of the Model

In [18]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()

# Neural Prophet

In [19]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [20]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date'])

# Start by cutting off data before 2020-01-06 and after 2025-03-02.
rs = rs[rs['created_date'] < '2025-03-02']
rs = rs[rs['created_date'] >= '2020-01-06']

## Restrict to STATEN ISLAND
rs = rs[rs['borough'] == 'STATEN ISLAND']

## Drop the column with borough
rs = rs.drop(columns=['borough'])

## Resample to weekly
rs_weekly = (
    rs.set_index('created_date')
      .resample('W')['count']
      .mean()
      .fillna(0)
      .reset_index()
      .rename(columns={'count': 'weekly_avg'})
)

## Rename columns for Prophet
rs_weekly.rename(columns={'created_date': 'ds', 'weekly_avg': 'y'}, inplace=True)

rs_weekly.head()

,ds,y
0,2020-01-12,2.250000
1,2020-01-19,1.500000
2,2020-01-26,1.333333
3,2020-02-02,2.000000
4,2020-02-09,1.500000


In [21]:
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd_daily = nd
else:
    wd_daily = pd.DataFrame(data["daily"])
    wd_daily["date"] = pd.to_datetime(wd_daily["time"])
    wd_daily = wd_daily.set_index("date")

# Resample to weekly (ending Sunday, to match rs_weekly)
wd = wd_daily.resample('W').agg({
    'temperature_2m_max':        'max',
    'temperature_2m_min':        'min',
    'temperature_2m_mean':       'mean',
    'apparent_temperature_max':  'max',
    'apparent_temperature_min':  'min',
    'apparent_temperature_mean': 'mean',
    'precipitation_sum':         'sum',
    'snowfall_sum':              'sum',
})

In [22]:
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd_merged = wd.reset_index().rename(columns={"date": "ds"})  # wd has a date index, not a 'time' column
wd_merged["ds"] = pd.to_datetime(wd_merged["ds"])
rs_weekly["ds"] = pd.to_datetime(rs_weekly["ds"])

rs_weekly = rs_weekly.merge(
    wd_merged[['ds'] + regressed_features],
    on="ds",
    how="left"
)

In [23]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=4)  # 4 weeks

def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 12),   # up to 12 weeks
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 12),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 4),                # up to 4 weeks
    }
    n_lags = trial.suggest_int('n_lags', 1, 12)                                 # up to 12 weeks
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 12, 248)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []

    for i, (train_idx, test_idx) in enumerate(tscv.split(rs_weekly)):

        train = rs_weekly.iloc[train_idx].copy()
        test = rs_weekly.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=False,  # no sub-weekly patterns in weekly data
            n_lags=n_lags,
            epochs=epochs,
            ar_reg=ar_reg,
            accelerator="auto",
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="W", progress="off")
        future = pd.concat([
            train[['ds', 'y'] + existing_regressors],
            test[['ds', 'y']].merge(wd_merged[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        intermediate_score = np.mean(fold_rmses)
        trial.report(intermediate_score, i)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=1)

best_params = study.best_params
print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-13 13:45:24,552] A new study created in memory with name: no-name-42a1ad5d-7272-473a-9000-b5ffb79cd1a0


Training: 0it [00:00, ?it/s]

Predicting: 4it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 4it [00:00, ?it/s]

[I 2026-03-13 13:45:39,900] Trial 0 finished with value: 0.9995680289861445 and parameters: {'lag_temp_max': 9, 'lag_temp_min': 2, 'lag_snowfall': 4, 'n_lags': 11, 'epochs': 176, 'learning_rate': 0.00507847531824012, 'batch_size': 75, 'ar_reg': 1.2306153792141576}. Best is trial 0 with value: 0.9995680289861445.


Best Parameters {'lag_temp_max': 9, 'lag_temp_min': 2, 'lag_snowfall': 4, 'n_lags': 11, 'epochs': 176, 'learning_rate': 0.00507847531824012, 'batch_size': 75, 'ar_reg': 1.2306153792141576}
Best RMSE: 0.9995680289861445


Parameters: {'lag_temp_max': 54, 'lag_temp_min': 18, 'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923}. Best is trial 83 with value: 2.9466650941279124.


In [24]:
# best_params = dict({'lag_temp_max': 54, 'lag_temp_min': 18, 
#                     'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923})

In [25]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date'])

# Start by cutting off data before 2020-01-06 and after 2025-03-02.
rs = rs[rs['created_date'] < '2025-03-02']
rs = rs[rs['created_date'] >= '2020-01-06']

## Restrict to STATEN ISLAND
rs = rs[rs['borough'] == 'STATEN ISLAND']

## Drop the column with borough
rs = rs.drop(columns=['borough'])

## Resample to weekly
rs_weekly = (
    rs.set_index('created_date')
      .resample('W')['count']
      .mean()
      .fillna(0)
      .reset_index()
      .rename(columns={'count': 'weekly_avg'})
)

## Rename columns for Prophet
rs_weekly.rename(columns={'created_date': 'ds', 'weekly_avg': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-06"
end   = "2025-03-02"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd_daily = pd.DataFrame(data["daily"])
wd_daily["date"] = pd.to_datetime(wd_daily["time"])
wd_daily = wd_daily.set_index("date")

# Resample to weekly (ending Sunday, to match rs_weekly)
wd = wd_daily.resample('W').agg({
    'temperature_2m_max':        'max',
    'temperature_2m_min':        'min',
    'temperature_2m_mean':       'mean',
    'apparent_temperature_max':  'max',
    'apparent_temperature_min':  'min',
    'apparent_temperature_mean': 'mean',
    'precipitation_sum':         'sum',
    'snowfall_sum':              'sum',
})

## Evaluate the Model

In [26]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=4)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd_merged = wd.reset_index().rename(columns={"date": "ds"})
wd_merged["ds"] = pd.to_datetime(wd_merged["ds"])
rs_weekly["ds"] = pd.to_datetime(rs_weekly["ds"])

rs_weekly = rs_weekly.merge(
    wd_merged[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']

results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs_weekly)):

    train = rs_weekly.iloc[train_index].copy()
    train = train.dropna(subset=["y"])
    test = rs_weekly.iloc[test_index].copy()

    model = NeuralProphet(
        yearly_seasonality=True,
        weekly_seasonality=False,          # no sub-weekly patterns in weekly data
        learning_rate=best_params['learning_rate'],
        epochs=best_params['epochs'],
        n_lags=best_params['n_lags'],
        ar_reg=best_params['ar_reg'],
        accelerator="auto",
        batch_size=best_params['batch_size']
    )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])

    model.fit(train, freq="W", progress="off")

    future = pd.concat([
        train[['ds', 'y'] + regressed_features],
        test[['ds', 'y']].merge(wd_merged[['ds'] + regressed_features], on="ds", how="left")
    ])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_val = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse_val, "mape": mape_val})

neural_prophet_weekly_results_df = pd.DataFrame(results)
neural_prophet_weekly_results_df.loc["mean"] = ["mean", neural_prophet_weekly_results_df["rmse"].mean(), neural_prophet_weekly_results_df["mape"].mean()]
neural_prophet_weekly_results_df


Training: 0it [00:00, ?it/s]

Predicting: 3it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 3it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Training: 0it [00:00, ?it/s]

Predicting: 4it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 4it [00:00, ?it/s]

,fold,rmse,mape
0,0,4.365419,2.667152
1,1,0.811456,0.403227
2,2,3.690018,0.681472
3,3,1.322330,0.554465
4,4,1.786769,0.389437
5,5,1.887854,0.380858
6,6,1.262494,0.408649
7,7,0.948359,0.277911
8,8,0.951589,0.380752
9,9,0.858883,0.487065


## Final Model

In [27]:
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']

model = NeuralProphet(
    yearly_seasonality=True,
    weekly_seasonality=False,          # no sub-weekly patterns in weekly data
    batch_size=best_params['batch_size'],
    ar_reg=best_params['ar_reg'],
    learning_rate=best_params['learning_rate'],
    epochs=best_params['epochs'],
    accelerator="auto",
    n_lags=best_params['n_lags']
)
model = model.add_country_holidays(country_name="US")
for column in regressed_features:
    model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])

model.fit(rs_weekly, freq="W", progress="off")

new_rs_weekly = rs_weekly.copy()
new_rs_weekly = new_rs_weekly.drop(columns=['y'])

forecast = model.predict(rs_weekly)

Training: 0it [00:00, ?it/s]

Predicting: 4it [00:00, ?it/s]

In [28]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
